In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/churn.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
TEST_SIZE = 0.25


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
print("churn rate:", round(data["churn"].mean(), 3))
print(data.isna().sum())


In [ ]:
# Распределение churn
churn_counts = data["churn"].value_counts().sort_index()
colors = sns.color_palette("rocket", 2)

plt.figure(figsize=(7, 6))
ax = plt.subplot(111)
wedges, _ = ax.pie(
    churn_counts,
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.45, edgecolor="w"),
)
legend_labels = [
    f"{'Остался' if i == 0 else 'Ушёл'} ({c} — {c / churn_counts.sum() * 100:.1f}%)"
    for i, c in churn_counts.items()
]
plt.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(1, 0.5))
plt.title("Доля оттока клиентов", fontsize=14, pad=15)
centre = plt.Circle((0, 0), 0.25, color="white")
ax.add_artist(centre)
plt.tight_layout()
plt.savefig(FIG / "churn_rate_pie.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Отток по типу контракта
contract_churn = data.groupby("contract_type")["churn"].mean().sort_values()

plt.figure(figsize=(8, 5))
sns.barplot(
    x=contract_churn.index,
    y=contract_churn.values,
    hue=contract_churn.index,
    palette="viridis",
    legend=False,
)
plt.title("Доля оттока по типу контракта", fontsize=14, pad=15)
plt.xlabel("Тип контракта", fontsize=12)
plt.ylabel("Доля churn", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)

for i, val in enumerate(contract_churn.values):
    plt.text(i, val + 0.01, f"{val:.2f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIG / "churn_by_contract.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Гистограмма monthly_charges с разделением по churn
plt.figure(figsize=(9, 5))
sns.histplot(
    data=data,
    x="monthly_charges",
    hue="churn",
    bins=30,
    kde=True,
    palette="rocket",
    alpha=0.6,
)
plt.title("Monthly charges: остались vs ушли", fontsize=14, pad=15)
plt.xlabel("Ежемесячный платёж", fontsize=12)
plt.ylabel("Количество", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "churn_monthly_charges_hist.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Тепловая карта корреляций числовых признаков
num_cols_corr = ["tenure_months", "monthly_charges", "support_tickets", "total_charges", "churn"]
corr = data[num_cols_corr].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,       # Подписи значений в ячейках
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
)
plt.title("Корреляции числовых признаков", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(FIG / "churn_corr_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 2. ПРИЗНАКИ И SPLIT
# ======================
num_cols = ["tenure_months", "monthly_charges", "support_tickets", "total_charges"]
cat_cols = ["contract_type", "internet_service"]
target = "churn"

X = data[num_cols + cat_cols]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
X_train.shape, X_test.shape


In [ ]:
# ======================
# 3. ПРЕПРОЦЕССИНГ
# ======================
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])


In [ ]:
# ======================
# 4. LOGISTIC REGRESSION
# ======================
logreg = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])
logreg.fit(X_train, y_train)
pred = logreg.predict(X_test)
proba = logreg.predict_proba(X_test)[:, 1]

print("LogReg AUC", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred, digits=3))


In [ ]:
# ======================
# 5. RANDOM FOREST
# ======================
rf = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED)),
])
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]

print("RF AUC", round(roc_auc_score(y_test, proba_rf), 3))
print(classification_report(y_test, pred_rf, digits=3))


In [ ]:
# ======================
# 6. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ======================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

RocCurveDisplay.from_predictions(y_test, proba, name="LogReg", ax=axes[0])
RocCurveDisplay.from_predictions(y_test, proba_rf, name="RF", ax=axes[0])
axes[0].set_title("ROC-кривые")
axes[0].grid(True, linestyle="--", alpha=0.5)

ConfusionMatrixDisplay.from_predictions(y_test, pred_rf, ax=axes[1], colorbar=False)
axes[1].set_title("RF — confusion matrix")

plt.tight_layout()
plt.savefig(FIG / "churn_roc.png", dpi=120, bbox_inches="tight")
plt.savefig(FIG / "churn_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Итог: сравниваем LogReg и RF по AUC + classification_report.")
print("Дальше можно смотреть precision/recall на бизнес-пороге.")
